In [0]:
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("schema_name", "foodlens_bronze")
dbutils.widgets.text("silver_schema", "foodlens_silver")

catalog_name  = dbutils.widgets.get("catalog_name")
schema_name   = dbutils.widgets.get("schema_name")
silver_schema = dbutils.widgets.get("silver_schema")

print("=" * 50)
print("PIPELINE PARAMETERS")
print("=" * 50)
print(f"Catalog       : {catalog_name}")
print(f"Bronze Schema : {schema_name}")
print(f"Silver Schema : {silver_schema}")
print("=" * 50)

In [0]:
%pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

Re-read parameters after restart

In [0]:
catalog_name  = dbutils.widgets.get("catalog_name")
schema_name   = dbutils.widgets.get("schema_name")
silver_schema = dbutils.widgets.get("silver_schema")

from pyspark.sql.functions import (
    col, lit, trim, upper, lower, when, regexp_extract, regexp_replace,
    split, explode, posexplode, size, expr, count, length, substring,
    monotonically_increasing_id, current_timestamp, concat_ws, array
)
from pyspark.sql.types import IntegerType, DoubleType, StringType
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient
import re

wc  = WorkspaceClient()
dqe = DQEngine(workspace_client=wc, spark=spark)

print(f"Parameters reloaded: {catalog_name}.{schema_name} -> {catalog_name}.{silver_schema}")
print("DQEngine initialized")

Load Bronze Tables

In [0]:
chicago_raw = spark.table(f"{catalog_name}.{schema_name}.chicago_inspections")
dallas_raw  = spark.table(f"{catalog_name}.{schema_name}.dallas_inspections")

print(f"Chicago Bronze: {chicago_raw.count():,} rows, {len(chicago_raw.columns)} cols")
print(f"Dallas Bronze:  {dallas_raw.count():,} rows, {len(dallas_raw.columns)} cols")

Step 1: Parse Chicago Violations String

In [0]:
# Chicago violations are pipe delimited: "code. DESCRIPTION ... Comments: ..." separated by " | "
# Split into individual violations, then extract code, description, comment from each

from pyspark.sql.functions import posexplode, trim, regexp_extract

# split the packed string on " | "
chi_violations_exploded = (
    chicago_raw
    .filter(col("Violations").isNotNull())
    .select(
        col("Inspection ID"),
        posexplode(split(col("Violations"), " \\| ")).alias("violation_seq", "raw_violation")
    )
)

# parse each chunk: code is the number before the dot, description is the text, comment is after "Comments:"
chi_violations_parsed = (
    chi_violations_exploded
    .withColumn("raw_violation", trim(col("raw_violation")))
    .withColumn("violation_code",
        regexp_extract(col("raw_violation"), r"^(\d+)\.", 1))
    .withColumn("violation_description",
        trim(regexp_extract(col("raw_violation"), r"^\d+\.\s*([^\-]+?)(?:\s*-\s*Comments:|$)", 1)))
    .withColumn("violation_comment",
        trim(regexp_extract(col("raw_violation"), r"Comments:\s*(.*)", 1)))
    .filter(col("violation_code") != "")
)

chi_violations_parsed.select("Inspection ID", "violation_seq", "violation_code", "violation_description", "violation_comment").show(10, truncate=80)
print(f"Chicago violations parsed: {chi_violations_parsed.count():,} rows from {chi_violations_exploded.count():,} exploded chunks")

Step 2: Unpivot Dallas Violation Columns

In [0]:
# Dallas has 25 slots: Violation Description/Points/Detail/Memo for each slot
# Unpivot all 25 into long format rows using SQL UNION ALL (serverless compatible)

dallas_core_cols = [
    "Restaurant Name", "Inspection Type", "Inspection Date", "Inspection Score",
    "Street Number", "Street Name", "Street Direction", "Street Type", "Street Unit",
    "Street Address", "Zip Code", "Inspection Month", "Inspection Year", "Lat Long Location",
    "_source_file", "_load_timestamp", "_source_city"
]

# register as temp view so we can use SQL
dallas_raw.createOrReplaceTempView("dallas_bronze")

# figure out which core cols actually exist
available_core = [c for c in dallas_core_cols if c in dallas_raw.columns]
core_select = ", ".join([f"`{c}`" for c in available_core])

# build UNION ALL across all 25 slots
union_parts = []
for i in range(1, 26):
    desc_col = f"Violation Description - {i}"
    pts_col  = f"Violation Points - {i}"
    det_col  = f"Violation Detail - {i}"
    memo_col = f"Violation Memo - {i}"

    # handle the typo column name (double space in Memo 20)
    if memo_col not in dallas_raw.columns:
        memo_col_alt = f"Violation  Memo - {i}"
        if memo_col_alt in dallas_raw.columns:
            memo_col = memo_col_alt

    if desc_col in dallas_raw.columns:
        pts_expr = f"CAST(`{pts_col}` AS INT)" if pts_col in dallas_raw.columns else "CAST(NULL AS INT)"
        det_expr = f"`{det_col}`" if det_col in dallas_raw.columns else "CAST(NULL AS STRING)"
        mem_expr = f"`{memo_col}`" if memo_col in dallas_raw.columns else "CAST(NULL AS STRING)"

        part = f"""
            SELECT {core_select},
                   {i} AS violation_seq,
                   `{desc_col}` AS violation_description,
                   {pts_expr} AS violation_points,
                   {det_expr} AS violation_detail,
                   {mem_expr} AS violation_memo
            FROM dallas_bronze
            WHERE `{desc_col}` IS NOT NULL"""
        union_parts.append(part)

full_sql = "\nUNION ALL\n".join(union_parts)
dallas_violations_long = spark.sql(full_sql)

print(f"Dallas violations unpivoted: {dallas_violations_long.count():,} rows")
dallas_violations_long.select("Restaurant Name", "Inspection Date", "violation_seq", "violation_description", "violation_points").show(10, truncate=60)

Step 3: Build Chicago Silver Table

In [0]:
# Derive inspection score from Results
# Standardize city/state casing
# Join violation count per inspection

chi_violation_counts = (
    chi_violations_parsed
    .groupBy("Inspection ID")
    .agg(
        count("*").alias("violation_count"),
        concat_ws(", ", array([lit("x")])).alias("_dummy")  # placeholder
    )
    .drop("_dummy")
)

chicago_silver = (
    chicago_raw
    .withColumn("inspection_score", 
        when(col("Results") == "Pass", 90)
        .when(col("Results") == "Pass w/ Conditions", 80)
        .when(col("Results") == "Fail", 70)
        .when(col("Results") == "No Entry", 0)
        .otherwise(lit(None).cast("int"))
    )
    .withColumn("City", upper(trim(col("City"))))
    .withColumn("State", upper(trim(col("State"))))
    .withColumn("source_city", lit("Chicago"))
)

# attach violation count
chicago_silver = chicago_silver.join(
    chi_violation_counts, on="Inspection ID", how="left"
).fillna(0, subset=["violation_count"])

print(f"Chicago Silver: {chicago_silver.count():,} rows")
chicago_silver.select("Inspection ID", "DBA Name", "Results", "inspection_score", "City", "State", "violation_count").show(10, truncate=False)

Step 4: Build Dallas Silver Table

In [0]:
# Count violations per inspection (using the long format)
dal_violation_counts = (
    dallas_violations_long
    .groupBy("Restaurant Name", "Inspection Date", "Inspection Score")
    .agg(count("*").alias("violation_count"))
)

dallas_silver = (
    dallas_raw
    .select(*[c for c in dallas_core_cols if c in dallas_raw.columns])
    .withColumn("source_city", lit("Dallas"))
)

# attach violation count
dallas_silver = dallas_silver.join(
    dal_violation_counts,
    on=["Restaurant Name", "Inspection Date", "Inspection Score"],
    how="left"
).fillna(0, subset=["violation_count"])

print(f"Dallas Silver: {dallas_silver.count():,} rows")
dallas_silver.select("Restaurant Name", "Inspection Date", "Inspection Score", "Zip Code", "violation_count").show(10, truncate=False)

Step 5: DQX Validation Rules (Assignment Requirements)

In [0]:
# Assignment rules using sql_expression for columns with spaces in names
# (DQX is_not_null doesn't handle spaces in column names)

chicago_rules = [
    {"name": "restaurant_name_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`DBA Name` IS NOT NULL",
         "msg": "Restaurant name cannot be null"}}},

    {"name": "inspection_date_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Date` IS NOT NULL",
         "msg": "Inspection date cannot be null"}}},

    {"name": "inspection_type_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Type` IS NOT NULL",
         "msg": "Inspection type cannot be null"}}},

    {"name": "zip_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Zip` IS NOT NULL",
         "msg": "Zip code cannot be null"}}},

    {"name": "results_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Results` IS NOT NULL",
         "msg": "Chicago inspection results cannot be null"}}},

    {"name": "has_at_least_1_violation", "criticality": "warn",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "violation_count >= 1",
         "msg": "Inspection has no violations"}}},
    
    {"name": "pass_no_urgent_critical", "criticality": "warn",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "NOT (Results = 'Pass' AND has_urgent_critical = true)",
         "msg": "Result cannot be PASS if violation contains Urgent/Critical"}}},
]

dallas_rules = [
    {"name": "restaurant_name_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Restaurant Name` IS NOT NULL",
         "msg": "Restaurant name cannot be null"}}},

    {"name": "inspection_date_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Date` IS NOT NULL",
         "msg": "Inspection date cannot be null"}}},

    {"name": "inspection_type_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Type` IS NOT NULL",
         "msg": "Inspection type cannot be null"}}},

    {"name": "zip_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Zip Code` IS NOT NULL",
         "msg": "Zip code cannot be null"}}},

    {"name": "zip_valid_format", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "CAST(`Zip Code` AS STRING) RLIKE '^[0-9]{5}(-[0-9]{4})?$'",
         "msg": "Zip code must be valid 5 or 9 digit format"}}},

    {"name": "score_not_over_100", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Score` <= 100",
         "msg": "Dallas violation score cannot exceed 100"}}},

    {"name": "has_at_least_1_violation", "criticality": "warn",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "violation_count >= 1",
         "msg": "Inspection has no violations"}}},

    {"name": "score_90_max_3_violations", "criticality": "warn",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "NOT (`Inspection Score` >= 90 AND violation_count > 3)",
         "msg": "Score >= 90 should not have more than 3 violations"}}},

    {"name": "pass_no_urgent_critical", "criticality": "warn",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "NOT (`Inspection Score` >= 90 AND has_urgent_critical = true)",
         "msg": "High score with Urgent/Critical violations"}}},
]

print(f"Chicago rules: {len(chicago_rules)}")
print(f"Dallas rules:  {len(dallas_rules)}")
for r in chicago_rules:
    print(f"  [CHI] {r['name']} ({r['criticality']})")
for r in dallas_rules:
    print(f"  [DAL] {r['name']} ({r['criticality']})")

Step 6: Run DQX Validation

In [0]:
print("=" * 60)
print("CHICAGO DQX VALIDATION")
print("=" * 60)

chi_good, chi_bad = dqe.apply_checks_by_metadata_and_split(chicago_silver, chicago_rules)

chi_total   = chicago_silver.count()
chi_passed  = chi_good.count()
chi_failed  = chi_bad.count()

print(f"Total:       {chi_total:,}")
print(f"Passed:      {chi_passed:,}")
print(f"Quarantined: {chi_failed:,}")

if chi_failed > 0:
    print("\nSample quarantined records:")
    chi_bad.select("Inspection ID", "DBA Name", "Results", "violation_count", "_errors", "_warnings").show(10, truncate=80)

In [0]:
print("=" * 60)
print("DALLAS DQX VALIDATION")
print("=" * 60)

dal_good, dal_bad = dqe.apply_checks_by_metadata_and_split(dallas_silver, dallas_rules)

dal_total   = dallas_silver.count()
dal_passed  = dal_good.count()
dal_failed  = dal_bad.count()

print(f"Total:       {dal_total:,}")
print(f"Passed:      {dal_passed:,}")
print(f"Quarantined: {dal_failed:,}")

if dal_failed > 0:
    print("\nSample quarantined records:")
    dal_bad.select("Restaurant Name", "Inspection Date", "Inspection Score", "violation_count", "_errors", "_warnings").show(10, truncate=80)

Step 7: Deduplicate Violations

In [0]:
# assignment says: duplication violations to be loaded as distinct

chi_violations_deduped = chi_violations_parsed.dropDuplicates(
    ["Inspection ID", "violation_code"]
)
print(f"Chicago violations: {chi_violations_parsed.count():,} -> {chi_violations_deduped.count():,} after dedup")

dallas_violations_deduped = dallas_violations_long.dropDuplicates(
    ["Restaurant Name", "Inspection Date", "violation_seq", "violation_description"]
)
print(f"Dallas violations: {dallas_violations_long.count():,} -> {dallas_violations_deduped.count():,} after dedup")

Step 8: Write Silver Tables

In [0]:
from pyspark.sql.functions import trim, col

# drop DQX internal columns before writing
drop_cols = ["_errors", "_warnings"]

# Chicago inspections
chi_clean = chi_good
for c in drop_cols:
    if c in chi_clean.columns:
        chi_clean = chi_clean.drop(c)

chi_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.chicago_inspections")
print(f"chicago_inspections written: {chi_clean.count():,} rows")

# Dallas inspections
dal_clean = dal_good
for c in drop_cols:
    if c in dal_clean.columns:
        dal_clean = dal_clean.drop(c)

dal_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.dallas_inspections")
print(f"dallas_inspections written: {dal_clean.count():,} rows")

# Chicago violations (deduped)
chi_violations_deduped.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.chicago_violations")
print(f"chicago_violations written: {chi_violations_deduped.count():,} rows")

# Dallas violations (deduped)
dallas_violations_deduped.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.dallas_violations")
print(f"dallas_violations written: {dallas_violations_deduped.count():,} rows")

# Quarantine tables (only records with actual errors, not just warnings)
from pyspark.sql.functions import size

chi_errors_only = chi_bad.filter(size(col("_errors")) > 0)
chi_errors_count = chi_errors_only.count()
if chi_errors_count > 0:
    chi_errors_only.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.columnMapping.mode", "name") \
        .option("delta.minReaderVersion", "2") \
        .option("delta.minWriterVersion", "5") \
        .saveAsTable(f"{catalog_name}.{silver_schema}.quarantine_chicago")
    print(f"quarantine_chicago written: {chi_errors_count:,} rows")
else:
    print("quarantine_chicago: no error records to quarantine")

dal_errors_only = dal_bad.filter(size(col("_errors")) > 0)
dal_errors_count = dal_errors_only.count()
if dal_errors_count > 0:
    dal_errors_only.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.columnMapping.mode", "name") \
        .option("delta.minReaderVersion", "2") \
        .option("delta.minWriterVersion", "5") \
        .saveAsTable(f"{catalog_name}.{silver_schema}.quarantine_dallas")
    print(f"quarantine_dallas written: {dal_errors_count:,} rows")
else:
    print("quarantine_dallas: no error records to quarantine")

Step 9: Verify Silver Tables

In [0]:
print("=" * 60)
print("SILVER TABLES")
print("=" * 60)
spark.sql(f"SHOW TABLES IN {catalog_name}.{silver_schema}").show(truncate=False)

print("\nChicago Inspections sample:")
spark.sql(f"SELECT `Inspection ID`, `DBA Name`, Results, inspection_score, City, violation_count FROM {catalog_name}.{silver_schema}.chicago_inspections LIMIT 5").show(truncate=False)

print("Dallas Inspections sample:")
spark.sql(f"SELECT `Restaurant Name`, `Inspection Date`, `Inspection Score`, `Zip Code`, violation_count FROM {catalog_name}.{silver_schema}.dallas_inspections LIMIT 5").show(truncate=False)

print("Chicago Violations sample:")
spark.sql(f"SELECT * FROM {catalog_name}.{silver_schema}.chicago_violations LIMIT 5").show(truncate=False)

print("Dallas Violations sample:")
spark.sql(f"SELECT `Restaurant Name`, violation_seq, violation_description, violation_points FROM {catalog_name}.{silver_schema}.dallas_violations LIMIT 5").show(truncate=60)

Summary

In [0]:
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)

chi_s = spark.table(f"{catalog_name}.{silver_schema}.chicago_inspections").count()
dal_s = spark.table(f"{catalog_name}.{silver_schema}.dallas_inspections").count()
chi_v = spark.table(f"{catalog_name}.{silver_schema}.chicago_violations").count()
dal_v = spark.table(f"{catalog_name}.{silver_schema}.dallas_violations").count()

print(f"Chicago inspections : {chi_s:,}")
print(f"Dallas inspections  : {dal_s:,}")
print(f"Chicago violations  : {chi_v:,} (deduped)")
print(f"Dallas violations   : {dal_v:,} (deduped)")
print(f"Chicago quarantined : {chi_errors_count:,}")
print(f"Dallas quarantined  : {dal_errors_count:,}")

print(f"""
Validation rules applied:
  Restaurant Name, Date, Type, Zip not null
  Chicago Results not null
  Dallas score <= 100
  Every inspection has >= 1 violation
  Dallas score >= 90 has <= 3 violations
  Result cannot be PASS with Urgent/Critical violations
  Violations deduped per inspection
  City/State casing standardized
  Chicago scores derived from Results
""")